In [ ]:
import os
import pandas as pd

# ========== 1. CONFIG ==========
# change this to your actual path (Colab: upload to /content/data or use drive)
DATA_DIR = "/content/sample_data/datasetproj"   # <-- change me

# how many rows to read from each big file (just for demo to supervisor)
SAMPLE_ROWS = 5000

# files we expect based on your screenshot
EXPECTED_FILES = [
    "logon.csv",
    "file.csv",
    "device.csv",
    "http.csv",
    "psychometric.csv"
    # if you later have "LDAP.csv" or "email.csv", you can add here
]

# ========== 2. LOAD CSVs (SAMPLED) ==========
loaded = {}
all_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv")]

print("CSV files found in folder:")
for f in all_files:
    print(" -", f)
print()

for f in all_files:
    path = os.path.join(DATA_DIR, f)
    name = f.rsplit(".", 1)[0].lower()  # e.g. "logon"
    # read with sampling
    try:
        df = pd.read_csv(path, nrows=SAMPLE_ROWS)
    except UnicodeDecodeError:
        # sometimes CERT files need latin-1
        df = pd.read_csv(path, nrows=SAMPLE_ROWS, encoding="latin-1")
    loaded[name] = df
    print(f"Loaded {f}: {df.shape[0]} rows, {df.shape[1]} columns")

print("\n=== COLUMN OVERVIEW ===")
for name, df in loaded.items():
    print(f"\n{name.upper()} ({df.shape[0]} x {df.shape[1]})")
    print(df.columns.tolist())

# ========== 3. BUILD SUMMARY TABLE ==========
summary_rows = []
for name, df in loaded.items():
    summary_rows.append({
        "dataset": name,
        "rows (sampled)": df.shape[0],
        "columns": df.shape[1],
        "sample_features": ", ".join(df.columns[:6])
    })
summary_df = pd.DataFrame(summary_rows)
print("\n=== DATASET SUMMARY TABLE ===")
print(summary_df)

# ========== 4. MERGE DATASETS ==========
# We will try to merge these in this order:
# start with LOGON -> merge FILE -> merge DEVICE -> merge HTTP -> add PSYCHOMETRIC
# If some file does not exist in your folder, we will skip it.

def has_df(name):
    return name in loaded and not loaded[name].empty

merged = None

# 4.1 start with logon
if has_df("logon"):
    merged = loaded["logon"].copy()
    # make sure there's a 'user' column name
    # sometimes CERT uses 'user' already; if not, adjust here
else:
    print("\n[WARN] logon.csv not found, starting with first available dataset")
    first_name = list(loaded.keys())[0]
    merged = loaded[first_name].copy()

# 4.2 merge file
if has_df("file"):
    if "user" in merged.columns and "user" in loaded["file"].columns:
        merged = merged.merge(loaded["file"], on="user", how="outer", suffixes=("", "_file"))
    else:
        print("[WARN] Could not merge FILE: 'user' column missing")

# 4.3 merge device
if has_df("device"):
    if "user" in merged.columns and "user" in loaded["device"].columns:
        merged = merged.merge(loaded["device"], on="user", how="outer", suffixes=("", "_device"))
    else:
        print("[WARN] Could not merge DEVICE: 'user' column missing")

# 4.4 merge http
if has_df("http"):
    if "user" in merged.columns and "user" in loaded["http"].columns:
        merged = merged.merge(loaded["http"], on="user", how="outer", suffixes=("", "_http"))
    else:
        print("[WARN] Could not merge HTTP: 'user' column missing")

# 4.5 merge psychometric (usually has user_id)
if has_df("psychometric"):
    psych_df = loaded["psychometric"]
    # try to join on user_id
    if "user" in merged.columns and "user_id" in psych_df.columns:
        merged = merged.merge(psych_df, left_on="user", right_on="user_id", how="left")
    elif "user" in merged.columns and "user" in psych_df.columns:
        merged = merged.merge(psych_df, on="user", how="left")
    else:
        print("[WARN] Could not merge PSYCHOMETRIC: no common column")

print("\n=== MERGED DATASET SHAPE ===")
print(merged.shape)

print("\n=== MERGED DATASET PREVIEW ===")
print(merged.head(20))

# ========== 5. SAVE (OPTIONAL) ==========
# You can save the merged sample as a CSV to show your supervisor.
OUTPUT_PATH = os.path.join(DATA_DIR, "merged_sample.csv")
merged.to_csv(OUTPUT_PATH, index=False)
print(f"\nMerged sample saved to: {OUTPUT_PATH}")


CSV files found in folder:
 - device (2).csv
 - 2010-01.csv
 - logon (2).csv
 - device.csv
 - psychometric.csv
 - http.csv
 - logon.csv
 - file.csv

Loaded device (2).csv: 5000 rows, 5 columns
Loaded 2010-01.csv: 998 rows, 5 columns
Loaded logon (2).csv: 5000 rows, 5 columns
Loaded device.csv: 5000 rows, 6 columns
Loaded psychometric.csv: 4000 rows, 7 columns
Loaded http.csv: 5000 rows, 5 columns
Loaded logon.csv: 5000 rows, 5 columns
Loaded file.csv: 5000 rows, 9 columns

=== COLUMN OVERVIEW ===

DEVICE (2) (5000 x 5)
['id', 'date', 'user', 'pc', 'activity']

2010-01 (998 x 5)
['employee_name', 'user_id', 'Domain', 'Email', 'Role']

LOGON (2) (5000 x 5)
['id', 'date', 'user', 'pc', 'activity']

DEVICE (5000 x 6)
['id', 'date', 'user', 'pc', 'file_tree', 'activity']

PSYCHOMETRIC (4000 x 7)
['employee_name', 'user_id', 'O', 'C', 'E', 'A', 'N']

HTTP (5000 x 5)
['{M8H9-W9NL75TH-1322KOLO}', '01/04/2010 07:08:47', 'DTAA/AMA0606', 'PC-1514', 'http://cnet.com']

LOGON (5000 x 5)
['id', 'dat

In [ ]:
# import os
# import pandas as pd

# # ========== 1. CONFIG ==========
# DATA_DIR = "/content/sample_data/datasetproj"   # <-- change this
# OUTPUT_PATH = os.path.join(DATA_DIR, "merged_full_with_labels.csv")

# # Memory-management options
# pd.options.mode.chained_assignment = None  # silence warnings
# CHUNKSIZE = None   # set to None to read all rows (you can change to 500000 if you hit memory errors)

# # ========== 2. LOAD ALL CSVs ==========
# loaded = {}
# csv_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv")]

# print("CSV files found:")
# for f in csv_files:
#     print(" -", f)

# def load_csv(fname):
#     path = os.path.join(DATA_DIR, fname)
#     print(f"\nLoading {fname} ...")
#     if CHUNKSIZE:
#         # stream read for huge files
#         chunks = pd.read_csv(path, chunksize=CHUNKSIZE, encoding="latin-1")
#         df = pd.concat(chunks, ignore_index=True)
#     else:
#         df = pd.read_csv(path, encoding="latin-1")
#     print(f"   {df.shape[0]:,} rows, {df.shape[1]} columns")
#     return df

# for f in csv_files:
#     key = f.split(".")[0].lower()
#     df = load_csv(f)
#     loaded[key] = df

# # ========== 3. MERGE MAIN DATASETS ==========
# def has_df(name): return name in loaded and not loaded[name].empty

# # Start with logon
# if has_df("logon"):
#     merged = loaded["logon"].copy()
# else:
#     first = list(loaded.keys())[0]
#     merged = loaded[first].copy()
#     print(f"[WARN] logon.csv missing; starting with {first}")

# def merge_on_user(base_df, add_df, suffix):
#     if "user" in base_df.columns and "user" in add_df.columns:
#         return base_df.merge(add_df, on="user", how="outer", suffixes=("", suffix))
#     else:
#         print(f"[WARN] Missing 'user' column when merging {suffix}")
#         return base_df

# for fkey in ["file", "device", "http"]:
#     if has_df(fkey):
#         merged = merge_on_user(merged, loaded[fkey], f"_{fkey}")

# # Add psychometric
# if has_df("psychometric"):
#     psych = loaded["psychometric"]
#     if "user" in merged.columns and "user_id" in psych.columns:
#         merged = merged.merge(psych, left_on="user", right_on="user_id", how="left")
#     elif "user" in merged.columns and "user" in psych.columns:
#         merged = merged.merge(psych, on="user", how="left")

# print("\nMerged shape (before labels):", merged.shape)

# # ========== 4. BUILD EVENT TIME ==========
# def build_event_time(df):
#     for c in ["date", "timestamp", "time", "datetime"]:
#         if c in df.columns:
#             return pd.to_datetime(df[c], errors="coerce")
#     return pd.to_datetime(pd.Series([None]*len(df)))

# merged["event_time"] = build_event_time(merged)

# # ========== 5. ADD INSIDER LABELS ==========
# insiders_df = None
# for k in loaded:
#     if "insider" in k:    # catch insiders.csv / answers.csv
#         insiders_df = loaded[k]
#         break

# merged["is_insider"] = 0

# if insiders_df is not None:
#     print("\nLabelling insiders ...")
#     insiders_df["start"] = pd.to_datetime(insiders_df["start"], errors="coerce")
#     insiders_df["end"] = pd.to_datetime(insiders_df["end"], errors="coerce")

#     for _, row in insiders_df.iterrows():
#         u = row["user"]
#         st, en = row["start"], row["end"]
#         mask_user = merged["user"] == u
#         if "event_time" in merged.columns and pd.notna(st) and pd.notna(en):
#             mask_time = (merged["event_time"] >= st) & (merged["event_time"] <= en)
#             merged.loc[mask_user & mask_time, "is_insider"] = 1
#         else:
#             merged.loc[mask_user, "is_insider"] = 1

#     print("Label counts:")
#     print(merged["is_insider"].value_counts())
# else:
#     print("\n[INFO] No insiders.csv / answers.csv found; skipping labelling")

# # ========== 6. SAVE MERGED DATA ==========
# print(f"\nSaving merged dataset to {OUTPUT_PATH} ...")
# merged.to_csv(OUTPUT_PATH, index=False)
# print("Done. Final shape:", merged.shape)

# # optional: preview
# print("\nPreview:")
# print(merged.head(10))
